In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Configuração para mostrar números sem notação científica (ex: 1,000,000.00)
pd.options.display.float_format = '{:,.2f}'.format

# --- FUNÇÕES DE LIMPEZA ---
def limpar_valor(valor):
    """Transforma '124M €' em 124000000.0"""
    if pd.isna(valor): return 0
    if isinstance(valor, (int, float)): return valor
    valor = str(valor).replace('€', '').strip()
    if 'M' in valor: return float(valor.replace('M', '')) * 1_000_000
    if 'K' in valor: return float(valor.replace('K', '')) * 1_000
    return float(valor)

def limpar_idade(valor):
    """Transforma '7 fev 1999 (26)' em 26"""
    if pd.isna(valor): return 0
    valor = str(valor)
    if '(' in valor:
        return int(valor.split('(')[-1].replace(')', ''))
    return 0

print("Bibliotecas e funções carregadas!")

Bibliotecas e funções carregadas!


In [ ]:
# 1. Carregar Jogadores (Verifique o nome do arquivo)
df = pd.read_csv('jogadores/sofascore_italiano_geral_final_valor_de_mercado_24_25.csv')
df['Liga'] = 'Serie A'

# 2. Aplicar Limpeza
df['Valor_Numerico'] = df['Valor de mercado'].apply(limpar_valor)
df['Idade_Limpa'] = df['Idade'].apply(limpar_idade)
df['Passe_Numerico'] = df['Acerto no passe %'].apply(lambda x: float(str(x).replace('%','')) if pd.notna(x) else 0)

# 3. CORREÇÃO DOS JOGADORES (O "Patch" Bundesliga)
# Mapeia jogadores para os times corretos da temporada 24/25
correcao_jogadores = {
    # --- BOLOGNA (A grande surpresa - Desmanche) ---
    'Joshua Zirkzee': 'Bologna',        # Foi pro Man Utd
    'Riccardo Calafiori': 'Bologna',    # Foi pro Arsenal
    'Lewis Ferguson': 'Bologna',        # Lesionado/Ficou
    'Alexis Saelemaekers': 'Bologna',   # Estava emprestado pelo Milan
    'Victor Kristiansen': 'Bologna',    # Estava emprestado
    'Marko Arnautović': 'Inter',        # Estava na Inter (mas pertenceu ao Bologna antes)

    # --- NAPOLI (Campeão 22/23 -> Crise 23/24) ---
    'Victor Osimhen': 'Napoli',         # Foi pro Galatasaray (Empréstimo)
    'Piotr Zieliński': 'Napoli',        # Foi pra Inter
    'Jesper Lindstrøm': 'Napoli',       # Foi pro Everton
    'Leo Østigård': 'Napoli',           # Foi pro Rennes
    'Natan': 'Napoli',                  # Foi pro Betis

    # --- JUVENTUS ---
    'Wojciech Szczęsny': 'Juventus',    # Aposentou/Saiu
    'Federico Chiesa': 'Juventus',      # Foi pro Liverpool
    'Adrien Rabiot': 'Juventus',        # Saiu (Marseille)
    'Alex Sandro': 'Juventus',          # Saiu (Flamengo)
    'Moise Kean': 'Juventus',           # Foi pra Fiorentina
    'Matías Soulé': 'Frosinone',        # Destaque no Frosinone (pertencia à Juve, foi pra Roma)
    'Dean Huijsen': 'Roma',             # Jogou na Roma (pertencia à Juve)
    'Kaio Jorge': 'Frosinone',          # Jogou no Frosinone (foi pro Cruzeiro)

    # --- INTER DE MILÃO (Campeã) ---
    'Alexis Sánchez': 'Inter',          # Saiu (Udinese)
    'Juan Cuadrado': 'Inter',           # Saiu (Atalanta)
    'Davy Klaassen': 'Inter',           # Saiu
    'Stefano Sensi': 'Inter',           # Saiu

    # --- MILAN ---
    'Olivier Giroud': 'Milan',          # Foi pra MLS
    'Simon Kjær': 'Milan',              # Saiu
    'Pierre Kalulu': 'Milan',           # Foi pra Juventus
    'Yacine Adli': 'Milan',             # Foi pra Fiorentina
    'Charles De Ketelaere': 'Atalanta', # Destaque na Atalanta (comprado em definitivo depois)

    # --- ROMA ---
    'Romelu Lukaku': 'Roma',            # Estava emprestado (foi pro Napoli)
    'Houssem Aouar': 'Roma',            # Foi pra Arábia
    'Leonardo Spinazzola': 'Roma',      # Foi pro Napoli
    'Rui Patrício': 'Roma',             # Foi pra Atalanta
    'Tammy Abraham': 'Roma',            # Foi pro Milan
    'Andrea Belotti': 'Roma',           # Foi pro Como

    # --- ATALANTA ---
    'Teun Koopmeiners': 'Atalanta',     # Foi pra Juve
    'El Bilal Touré': 'Atalanta',       # Foi pro Stuttgart
    'Juan Musso': 'Atalanta',           # Foi pro Atlético Madrid
    'Aleksei Miranchuk': 'Atalanta',    # Foi pra MLS

    # --- OUTROS DESTAQUES ---
    'Ciro Immobile': 'Lazio',           # Lenda da Lazio (foi pro Besiktas)
    'Luis Alberto': 'Lazio',            # Foi pro Catar
    'Felipe Anderson': 'Lazio',         # Foi pro Palmeiras
    'Albert Guðmundsson': 'Genoa',      # Foi pra Fiorentina
    'Mateo Retegui': 'Genoa',           # Foi pra Atalanta
    'Josep Martínez': 'Genoa',          # Foi pra Inter
    'Andrea Colpani': 'Monza',          # Foi pra Fiorentina
    'Valentín Carboni': 'Monza',        # Foi pro Marselha
    'Raoul Bellanova': 'Torino',        # Foi pra Atalanta
    'Alessandro Buongiorno': 'Torino'   # Foi pro Napoli
}

# Aplica a correção e salva o nome antigo em 'Time_Original' para conferênci
df['Time_Original'] = df['Time']
df['Time'] = df['Nome'].map(correcao_jogadores).fillna(df['Time'])

print(f"Jogadores carregados: {len(df)}")


Jogadores carregados: 330


In [ ]:
# 1. Carregar Tabela
df_tabela = pd.read_csv('tabela/tabela_serie_a_24_25.csv')

# 2. Limpeza de Colunas (Remove espaços extras nos nomes das colunas)
df_tabela.columns = df_tabela.columns.str.strip()

# 3. Renomear para 'Pos' (Garante que o código ache a coluna)
mapa_colunas = {'Posição': 'Pos', '#': 'Pos', 'Rank': 'Pos', 'RK': 'Pos'}
df_tabela.rename(columns=mapa_colunas, inplace=True)

# 4. Tratar Gols (XX:YY) para Gols Pró e Contra
if 'Gols' in df_tabela.columns:
    gols_split = df_tabela['Gols'].str.split(':', expand=True)
    if len(gols_split.columns) == 2:
        df_tabela['Gols_Pro'] = pd.to_numeric(gols_split[0])
        df_tabela['Gols_Contra'] = pd.to_numeric(gols_split[1])

# 5. CORREÇÃO DOS NOMES DOS TIMES (Para bater com os Jogadores)
# Esquerda: Nome na Tabela | Direita: Nome no arquivo de Jogadores (SofaScore)
mapa_times_tabela = {
    # Inter
    'Inter': 'Inter',
    'Internazionale': 'Inter',
    
    # Milan
    'Milan': 'Milan',
    'AC Milan': 'Milan',
    
    # Juventus
    'Juventus': 'Juventus',
    'Juve': 'Juventus',
    
    # Atalanta
    'Atalanta': 'Atalanta',
    'Atalanta BC': 'Atalanta',
    
    # Bologna
    'Bologna': 'Bologna',
    'Bologna FC': 'Bologna',
    
    # Roma
    'Roma': 'Roma',
    'AS Roma': 'Roma',
    
    # Lazio
    'Lazio': 'Lazio',
    'SS Lazio': 'Lazio',
    
    # Fiorentina
    'Fiorentina': 'Fiorentina',
    'ACF Fiorentina': 'Fiorentina',
    
    # Torino
    'Torino': 'Torino',
    'Torino FC': 'Torino',
    
    # Napoli
    'Napoli': 'Napoli',
    'SSC Napoli': 'Napoli',
    
    # Genoa
    'Genoa': 'Genoa',
    
    # Monza
    'Monza': 'Monza',
    
    # Hellas Verona
    'Hellas Verona': 'Hellas Verona',
    'Verona': 'Hellas Verona',
    
    # Lecce
    'Lecce': 'Lecce',
    
    # Udinese
    'Udinese': 'Udinese',
    
    # Cagliari
    'Cagliari': 'Cagliari',
    
    # Empoli
    'Empoli': 'Empoli',
    
    # Times Rebaixados 23/24 (Frosinone, Sassuolo, Salernitana)
    'Frosinone': 'Frosinone',
    'Sassuolo': 'Sassuolo',
    'Salernitana': 'Salernitana',
    
    # Times que Subiram (Parma, Como, Venezia)
    'Parma': 'Parma',
    'Como': 'Como',
    'Venezia': 'Venezia'
}
df_tabela['Time'] = df_tabela['Time'].str.strip().replace(mapa_times_tabela)

elenco_stats = df.groupby('Time').agg({
    'Valor_Numerico': 'sum',
    'Média das notas Sofascore': 'mean'
}).reset_index().rename(columns={'Valor_Numerico': 'Valor_Total_Elenco', 'Média das notas Sofascore': 'Media_Sofa_Time'})

df_times_final = pd.merge(df_tabela, elenco_stats, on='Time', how='inner')

df_final = pd.merge(df, df_times_final[['Time', 'Pos', 'PTS', 'Media_Sofa_Time', 'Valor_Total_Elenco']], on='Time', how='inner')

print(f"Merge concluído! Total de jogadores com dados completos: {len(df_final)}")

Merge concluído! Total de jogadores com dados completos: 256


In [5]:
# 1. CÁLCULO DO ÍNDICE ATLAS (Performance Individual vs Coletiva)
df_final['Indice_Atlas'] = df_final['Média das notas Sofascore'] - df_final['Media_Sofa_Time']

print("=== 🏋️ ÍNDICE ATLAS: QUEM CARREGA O TIME? ===")
display(df_final.sort_values('Indice_Atlas', ascending=False)[['Nome', 'Time', 'Pos', 'Média das notas Sofascore', 'Indice_Atlas']].head(10))

# 2. ANÁLISE DE CONTEXTO (Luxo na Lama)
print("\n=== 💸 LUXO DESPERDIÇADO (Jogadores Caros em Times da Parte de Baixo) ===")
# Critério: Valor > 20M e Posição > 9
luxo = df_final[
    (df_final['Valor_Numerico'] > 20_000_000) & 
    (df_final['Pos'] > 9)
].sort_values('Valor_Numerico', ascending=False)

if len(luxo) > 0:
    display(luxo[['Nome', 'Time', 'Pos', 'Valor de mercado', 'Média das notas Sofascore']])
else:
    print("Nenhum jogador encontrado com esses critérios na Bundesliga.")

# 3. ANÁLISE DE CONTEXTO (Diamantes na Lama)
print("\n=== 💎 DIAMANTES (Jogadores de Elite em Times da Parte de Baixo) ===")
# Critério: Nota > 7.15 e Posição > 9
diamantes = df_final[
    (df_final['Média das notas Sofascore'] > 7.15) & 
    (df_final['Pos'] > 9)
].sort_values('Média das notas Sofascore', ascending=False)

display(diamantes[['Nome', 'Time', 'Pos', 'Valor de mercado', 'Média das notas Sofascore']])

=== 🏋️ ÍNDICE ATLAS: QUEM CARREGA O TIME? ===


,Nome,Time,Pos,Média das notas Sofascore,Indice_Atlas
17,Paulo Dybala,Roma,5,7.39,0.45
149,Oumar Solet,Udinese,12,7.23,0.45
55,Mateo Retegui,Genoa,13,7.33,0.43
35,Mattia Zaccagni,Lazio,7,7.34,0.42
123,Riccardo Orsolini,Bologna,9,7.25,0.35
124,Alexis Saelemaekers,Bologna,9,7.25,0.35
166,Elia Caprile,Cagliari,15,7.17,0.34
0,Hakan Çalhanoğlu,Inter,2,7.41,0.33
68,Ademola Lookman,Atalanta,3,7.32,0.32
138,Nico Paz,Como,10,7.23,0.30



=== 💸 LUXO DESPERDIÇADO (Jogadores Caros em Times da Parte de Baixo) ===


,Nome,Time,Pos,Valor de mercado,Média das notas Sofascore
138,Nico Paz,Como,10,59M €,7.23
179,Alessandro Buongiorno,Torino,11,53M €,7.16
55,Mateo Retegui,Genoa,13,44M €,7.33
181,Raoul Bellanova,Torino,11,23M €,7.00
227,Zion Suzuki,Parma,16,21M €,7.03



=== 💎 DIAMANTES (Jogadores de Elite em Times da Parte de Baixo) ===


,Nome,Time,Pos,Valor de mercado,Média das notas Sofascore
55,Mateo Retegui,Genoa,13,44M €,7.33
138,Nico Paz,Como,10,59M €,7.23
149,Oumar Solet,Udinese,12,16.9M €,7.23
166,Elia Caprile,Cagliari,15,10.4M €,7.17
179,Alessandro Buongiorno,Torino,11,53M €,7.16


In [ ]:
def ficha_jogador(parte_do_nome):
    """
    Busca um jogador pelo nome (ou parte dele) e mostra todas as colunas.
    """
    dataframe_alvo = df 
    
    # 2. Faz a busca (converte tudo para minúsculo para não ter erro de maiúscula/minúscula)
    mask = dataframe_alvo['Nome'].str.lower().str.contains(parte_do_nome.lower(), na=False)
    resultado = dataframe_alvo[mask]
    
    # 3. Verifica se achou
    if len(resultado) == 0:
        return f"❌ Nenhum jogador encontrado com o nome '{parte_do_nome}'."
    

    return resultado.T


nome_para_buscar = input("Digite o nome (ou parte do nome) do jogador: ")

display(ficha_jogador(nome_para_buscar))

,3
Link Jogador,https://www.sofascore.com/pt/football/player/m...
Time,Genoa
Valor de mercado,44M €
Idade,29 de abril de 1999 (26)
Pontos fortes,Posicionamento; Jogador âncora; Alta pressão
Pontos fracos,Controle de bola
Posição,Forward
Gols,25
Gols esperados (xG),19.24
Dribles certos,12


In [ ]:
# Preparando dados para o gráfico
grafico_df = df_times_final.copy()

# Tratamento para evitar erro no gráfico (Divisão por zero ou NaN)
grafico_df['PTS'] = pd.to_numeric(grafico_df['PTS'], errors='coerce').fillna(0)
grafico_df['Valor_Total_Elenco'] = pd.to_numeric(grafico_df['Valor_Total_Elenco'], errors='coerce').fillna(0)
pts_ajustado = grafico_df['PTS'].replace(0, 1) # Evita dividir por zero

grafico_df['Custo_Por_Ponto'] = grafico_df['Valor_Total_Elenco'] / pts_ajustado

fig = px.scatter(grafico_df, 
                 x='Valor_Total_Elenco', 
                 y='PTS', 
                 text='Time', 
                 size='Custo_Por_Ponto', 
                 color='Pos',
                 color_continuous_scale='Bluered_r', # Azul = Topo, Vermelho = Baixo
                 hover_data=['Custo_Por_Ponto'],
                 title='Eficiência Financeira Bundesliga: Valor do Elenco x Pontos',
                 labels={'Valor_Total_Elenco': 'Investimento Total (€)', 'PTS': 'Pontos Conquistados'})

fig.update_traces(textposition='top center')

fig.update_yaxes(range=[0, 100], autorange=False, dtick=10)
fig.show()

In [ ]:
def scouting_serie_a(posicao=None, orcamento_max=None, atributo_obrigatorio=None):
    res = df_final.copy()
    
    if posicao:
        res = res[res['Posição'].str.contains(posicao, case=False, na=False)]
    
    if orcamento_max:
        valor = float(orcamento_max) * 1_000_000
        res = res[res['Valor_Numerico'] <= valor]
        
    if atributo_obrigatorio:
        res = res[res['Pontos fortes'].str.contains(atributo_obrigatorio, case=False, na=False)]
        
    cols = ['Nome', 'Time', 'Idade_Limpa', 'Valor de mercado', 'Média das notas Sofascore', 'Pontos fortes']
    return res.sort_values('Média das notas Sofascore', ascending=False)[cols].head(10)


print("🔍 Resultado Scouting:")
display(scouting_serie_a(posicao='Midfielder', orcamento_max=50))

🔍 Resultado Scouting (Meias, Max 40M, Bom de Passe):


,Nome,Time,Idade_Limpa,Valor de mercado,Média das notas Sofascore,Pontos fortes
0,Hakan Çalhanoğlu,Inter,31,26M €,7.41,Marcador de pênalti; Armador; Consistência
35,Mattia Zaccagni,Lazio,30,18M €,7.34,Passes longos; Duelos no chão; Armador
86,Scott McTominay,Napoli,28,NaN,7.27,Passes longos
112,Manuel Locatelli,Juventus,27,29M €,7.25,Passes longos; Posicionamento; Duelos no chão
123,Riccardo Orsolini,Bologna,28,26M €,7.25,Sem pontos fortes notáveis
124,Alexis Saelemaekers,Bologna,26,21M €,7.25,Armador; Passe; Passes longos
2,Federico Dimarco,Inter,28,48M €,7.24,Passe; Armador
71,Marten de Roon,Atalanta,34,4.8M €,7.16,Posicionamento
70,Éderson,Atalanta,26,43M €,7.16,Duelos no chão; Passes longos; Posicionamento
139,Lucas Da Cunha,Como,24,9.2M €,7.13,Armador; Finalizações de longe; Alta pressão
